# Agent Development Notebook

Interactive notebook for testing agent nodes as they are implemented.

## Prerequisites
```bash
cd ~/devops-ai-agentic && git pull
```

In [ ]:
!pip install -r /opt/app-root/src/devops-ai-agentic/agent/requirements.txt

In [ ]:
# MUST run before any other import
import pysqlite3
import sys
sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")
sys.path.insert(0, "/opt/app-root/src/devops-ai-agentic")
print("Setup complete")

## Story 2.3 — RAG Knowledge Base

In [ ]:
from agent.knowledge import build_index, search_knowledge

print("Building index (downloads nomic-embed-text-v1 ~547 MB on first run)...")
build_index()
print("Index ready.")

In [ ]:
results = search_knowledge("container cannot pull image from registry", n_results=2)
for r in results:
    print(r[:400])
    print("---")

In [ ]:
results = search_knowledge("secret db-password not found", alert_type="MISSING_SECRET", n_results=1)
for r in results:
    print(r[:400])

## Story 2.4 — monitor_cluster node

In [ ]:
from agent.nodes.monitor_cluster import monitor_cluster, WATCHED_NAMESPACES

print(f"Watched namespaces: {WATCHED_NAMESPACES}")

result = monitor_cluster({})
alerts = result["alerts"]
print(f"Alerts found: {len(alerts)}")
for alert in alerts:
    print(f"  [{alert['namespace']}/{alert['pod']}] {alert['reason']}: {alert['message'][:120]}")

In [ ]:
# Deploy a broken pod to test detection
import subprocess
subprocess.run([
    "oc", "run", "broken-pod", "-n", "ai-agentic",
    "--image=registry.example.com/nonexistent:v999",
    "--restart=Never",
], check=False)
print("Broken pod created — wait ~30s then re-run the cell above")

In [ ]:
# Clean up
subprocess.run(["oc", "delete", "pod", "broken-pod", "-n", "ai-agentic"], check=False)
print("Cleaned up")

## Full Graph

## Story 2.5 — classify_alert node

In [ ]:
import os
os.environ["QWEN_INFERENCE_URL"] = "http://qwen-predictor.ai-agentic.svc.cluster.local:8080"

from agent.nodes.classify_alert import classify_alert

# Test 1 — ImagePullBackOff (fast-path, no LLM call)
state = {"current_alert": {"reason": "ImagePullBackOff", "message": "pull access denied for registry.example.com/myapp:v2"}}
result = classify_alert(state)
print(f"Test 1 ImagePullBackOff : {result['alert_type']}")

# Test 2 — Missing secret (fast-path, no LLM call)
state = {"current_alert": {"reason": "CreateContainerConfigError", "message": "secret \"db-password\" not found"}}
result = classify_alert(state)
print(f"Test 2 MissingSecret    : {result['alert_type']}")

# Test 3 — Ambiguous alert (LLM call to Qwen)
state = {"current_alert": {"reason": "CrashLoopBackOff", "message": "container exited with code 1"}}
result = classify_alert(state)
print(f"Test 3 CrashLoopBackOff : {result['alert_type']}")

## Story 2.15 — investigate_image node (IMAGE_PULL_BACKOFF)

Reads the current (broken) container image from the Deployment and the last working image
from ReplicaSet rollout history, then fetches recent Helm file diffs from GitHub.

**Setup — trigger an ImagePullBackOff on demo-todo** from your local terminal:
```bash
# Patch demo-todo with a non-existent image tag
oc set image deployment/demo-todo demo-todo=quay.io/nonexistent/demo-todo:broken -n agent-apps
# Wait ~30 s for the pod to enter ImagePullBackOff
oc get pods -n agent-apps -w
```

**Teardown** (run after testing):
```bash
# Revert to the working image — check the previous tag first with:
# oc rollout history deployment/demo-todo -n agent-apps
oc rollout undo deployment/demo-todo -n agent-apps
```

In [ ]:
import os
os.environ.setdefault("GITHUB_TOKEN", "")           # set your token if rate-limiting is a concern
os.environ.setdefault("REPO_MAPPING_FILE", "/opt/app-root/src/devops-ai-agentic/k8s/ai-agentic/gpu/agent-repo-mapping.yaml")

from agent.nodes.investigate_image import investigate_image

# Simulates the state after monitor_cluster + classify_alert
# Replace pod name with an actual ImagePullBackOff pod from: oc get pods -n agent-apps
state = {
    "current_alert": {
        "namespace": "agent-apps",
        "pod": "demo-todo-<replace-with-real-suffix>",   # e.g. demo-todo-7d9f6b8c4-xk2vp
        "reason": "ImagePullBackOff",
        "message": "Back-off pulling image",
    },
}

result = investigate_image(state)
inv = result["investigation"]
print(f"current_image  : {inv.get('current_image')}")
print(f"previous_image : {inv.get('previous_image')}")
print(f"helm_repo      : {inv.get('helm_repo')}")
print(f"diff_chars     : {len(inv.get('helm_diff', ''))}")
print()
if inv.get("helm_diff"):
    print("--- Helm diff (truncated) ---")
    print(inv["helm_diff"][:800])

## Story 2.15 — investigate_code node (CRASH_LOOP)

Fetches the last 20 lines of logs from a crashing pod, then fetches recent source code
diffs from GitHub (excludes Helm/docs files). No automated fix is applied — the output
enriches the LLM report with root cause context.

**Setup — trigger a CrashLoopBackOff on demo-todo** from your local terminal:
```bash
# Point the app at a non-existent secret to make the container crash on startup
oc set env deployment/demo-todo CRASH_TEST=1 -n agent-apps
# OR deploy the missing-secret scenario values:
# oc apply -f /path/to/demo-todo/helm/values-scenario-missingsecret.yaml ...
# Wait ~30 s
oc get pods -n agent-apps -w
```

**Teardown**:
```bash
oc set env deployment/demo-todo CRASH_TEST- -n agent-apps   # removes the env var
# or simply: oc rollout undo deployment/demo-todo -n agent-apps
```

In [ ]:
from agent.nodes.investigate_code import investigate_code

# Replace pod name with an actual CrashLoopBackOff pod from: oc get pods -n agent-apps
state = {
    "current_alert": {
        "namespace": "agent-apps",
        "pod": "demo-todo-<replace-with-real-suffix>",   # e.g. demo-todo-7d9f6b8c4-xk2vp
        "reason": "CrashLoopBackOff",
        "message": "Back-off restarting failed container",
    },
}

result = investigate_code(state)
inv = result["investigation"]
print(f"source_repo    : {inv.get('source_repo')}")
print(f"log_lines      : {len(inv.get('pod_logs', '').splitlines())}")
print(f"diff_chars     : {len(inv.get('code_diff', ''))}")
print()
if inv.get("pod_logs"):
    print("--- Pod logs (last 20 lines) ---")
    print(inv["pod_logs"])
if inv.get("code_diff"):
    print()
    print("--- Recent source code changes (truncated) ---")
    print(inv["code_diff"][:800])

### Story 2.15 — investigate_image → search_rag → plan_fix (chained)

Tests the full IMAGE_PULL_BACKOFF pipeline in one shot.
`plan_fix` should pick up `previous_image` from the investigation and propose
a `patch_deployment_image` action using that exact tag.

In [ ]:
import json
from agent.nodes.investigate_image import investigate_image
from agent.nodes.search_rag import search_rag
from agent.nodes.plan_fix import plan_fix

# Replace with a real pod name from: oc get pods -n agent-apps
state = {
    "current_alert": {
        "namespace": "agent-apps",
        "pod": "demo-todo-<replace-with-real-suffix>",
        "reason": "ImagePullBackOff",
        "message": "Back-off pulling image quay.io/nonexistent/demo-todo:broken",
    },
    "alert_type": "IMAGE_PULL_BACKOFF",
}

# Step 1: investigate
state.update(investigate_image(state))
inv = state["investigation"]
print(f"[investigate_image] current={inv.get('current_image')!r}  previous={inv.get('previous_image')!r}")

# Step 2: search RAG
state.update(search_rag(state))
print(f"[search_rag] solutions={len(state['solutions'])}")

# Step 3: plan fix
state.update(plan_fix(state))
print(f"[plan_fix] fix_plan:")
print(json.dumps(json.loads(state["fix_plan"]), indent=2) if state.get("fix_plan") else "EMPTY")

## Story 2.6 — search_rag node

In [ ]:
from agent.nodes.search_rag import search_rag

# Test 1 — IMAGE_PULL_BACKOFF: should return KB-0004 runbook
state = {
    "current_alert": {"reason": "ImagePullBackOff", "message": "pull access denied for registry.example.com/myapp:v2"},
    "alert_type": "IMAGE_PULL_BACKOFF",
}
result = search_rag(state)
print(f"Solutions found: {len(result['solutions'])}")
for i, s in enumerate(result["solutions"], 1):
    print(f"\n--- Result {i} ---")
    print(s[:300])

## Story 2.7 — plan_fix node

In [ ]:
import json
from agent.nodes.plan_fix import plan_fix
from agent.nodes.search_rag import search_rag

# Test 1 — IMAGE_PULL_BACKOFF
state = {
    "current_alert": {
        "pod": "myapp-abc123",
        "namespace": "ai-agentic",
        "reason": "ImagePullBackOff",
        "message": "pull access denied for registry.example.com/myapp:v2",
    },
    "alert_type": "IMAGE_PULL_BACKOFF",
    "solutions": [],
}
state.update(search_rag(state))  # enrich with RAG context
result = plan_fix(state)
print("Test 1 fix plan:")
print(json.dumps(json.loads(result["fix_plan"]), indent=2) if result["fix_plan"] else "EMPTY")

print()

# Test 2 — MISSING_SECRET
state = {
    "current_alert": {
        "pod": "myapp-xyz789",
        "namespace": "ai-agentic",
        "reason": "CreateContainerConfigError",
        "message": 'secret "db-password" not found',
    },
    "alert_type": "MISSING_SECRET",
    "solutions": [],
}
state.update(search_rag(state))
result = plan_fix(state)
print("Test 2 fix plan:")
print(json.dumps(json.loads(result["fix_plan"]), indent=2) if result["fix_plan"] else "EMPTY")

## Story 2.8 — execute_fix node

> **Before running Test 1**, create a `myapp` deployment from your **local terminal**:
> ```bash
> oc create deployment myapp --image=nginx:latest -n ai-agentic
> ```
> Test 1 patches this deployment's image. Without it you'll get a 404.
>
> Tests 2 and 3 need no setup — Test 2 creates a secret (idempotent), Test 3 verifies the guardrail blocks system namespaces.

In [ ]:
from agent.nodes.execute_fix import execute_fix

# Test 1 — patch_deployment_image (needs a real deployment named "myapp" in ai-agentic)
# Create a test deployment first from your LOCAL terminal:
#   oc create deployment myapp --image=nginx:latest -n ai-agentic
state = {
    "current_alert": {"namespace": "ai-agentic"},
    "fix_plan": '{"action": "patch_deployment_image", "target": "myapp", "params": {"image": "nginx:1.25"}}',
}
result = execute_fix(state)
print(f"Test 1: {result['fix_result']}")

print()

# Test 2 — create_secret
state = {
    "current_alert": {"namespace": "ai-agentic"},
    "fix_plan": '{"action": "create_secret", "target": "test-secret", "params": {"data": {"key": "value123"}}}',
}
result = execute_fix(state)
print(f"Test 2: {result['fix_result']}")

print()

# Test 3 — guardrail blocks system namespace
state = {
    "current_alert": {"namespace": "openshift-monitoring"},
    "fix_plan": '{"action": "create_secret", "target": "evil", "params": {"data": {"key": "val"}}}',
}
result = execute_fix(state)
print(f"Test 3 (guardrail): {result['fix_result']}")

## Story 2.9 — verify node

> **Note:** These tests are isolated — they test `verify` standalone, not as part of the full graph.
> `verify` polls for a pod by name, so you need to pre-create one to simulate a successful fix.
>
> **Before running Test 1**, create a healthy pod from your **local terminal**:
> ```bash
> oc run healthy-pod --image=nginx:latest -n ai-agentic --restart=Never
> # wait for it to be Running
> oc get pod healthy-pod -n ai-agentic -w
> ```
> **After testing**, clean up:
> ```bash
> oc delete pod healthy-pod -n ai-agentic
> ```
>
> Tests 2 and 3 use a non-existent pod name on purpose — timeout and max-retries behaviour is the expected result.

In [ ]:
from agent.nodes.verify import verify

# Test 1 — healthy pod (create a running pod first from LOCAL terminal):
#   oc run healthy-pod --image=nginx:latest -n ai-agentic --restart=Never
state = {
    "current_alert": {"pod": "healthy-pod", "namespace": "ai-agentic"},
    "retry_count": 0,
}
result = verify(state)
print(f"Test 1 verified={result['verified']} retry_count={result['retry_count']}")

print()

# Test 2 — non-existent pod (simulates fix not yet applied)
state = {
    "current_alert": {"pod": "nonexistent-pod-abc123", "namespace": "ai-agentic"},
    "retry_count": 0,
}
result = verify(state)
print(f"Test 2 verified={result['verified']} retry_count={result['retry_count']}")

print()

# Test 3 — max retries reached
state = {
    "current_alert": {"pod": "nonexistent-pod-abc123", "namespace": "ai-agentic"},
    "retry_count": 2,  # already at MAX_RETRIES - 1
}
result = verify(state)
print(f"Test 3 (max retries) verified={result['verified']} retry_count={result['retry_count']}")

## Story 2.10 — report node

In [ ]:
# Run the full agent graph end-to-end
# Requires: a broken pod in ai-agentic namespace, QWEN_INFERENCE_URL set
result = graph.invoke({})
print(result.get("report", "No report generated"))

In [ ]:
from agent.graph import graph
from IPython.display import Image

Image(graph.get_graph().draw_mermaid_png())

In [ ]:
# Run the full graph (uncomment when all nodes are implemented)
# result = graph.invoke({})
# print(result.get("report", "No report generated"))